Takes about 50s to load TTS model

In [1]:
from itertools import islice

import litellm

from main import read_jsonl, TranslationOutput, message_content_from
from prompts import CAPTION_PROMPT


/home/maria/BookAudio/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/bin/sh: 1: sox: not found
SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 922.28it/s]


In [2]:

# sentences = list(islice((x for x in read_jsonl("out.jsonl") if x["type"] == "sentence"), 5))

# # for (i, sentence) in enumerate(sentences):
# #     generate_speech(sentence["content"], f"sentence-{i}.wav")

# response = litellm.completion(
#     model="openai/unsloth/Qwen3.6-35B-A3B-MTP-GGUF:UD-Q4_K_M",
#     api_base="http://localhost:8080/v1",
#     api_key="not-needed",
#     temperature=0.0,
#     messages=[{
#         "role": "system",
#         "content": CAPTION_PROMPT.format(
#             source_language="English",
#             target_language="Russian",
#         ),
#     }, {
#         "role": "user",
#         "content": f"""
# <context>
# {" ".join([sentence["content"] for sentence in sentences])}
# </context>

# <target>
# {sentences[3]["content"]}
# </target>
#         """
#     }],
#     response_format=TranslationOutput,
# )

# output = TranslationOutput.model_validate_json(
#     message_content_from(llm_response=response) or ""
# )
# output


In [3]:
output = TranslationOutput.model_validate({
    'sentences': [{
        'source': 'Many years later, Meggie had only to close her eyes',
        'target': 'Много лет спустя Мегги стоило только закрыть глаза',
        'uncommon_words': []
    },{
        'source': 'and she could still hear it,',
        'target': 'и она всё ещё могла слышать это,',
        'uncommon_words': []
    }, {
        'source': 'like tiny fingers tapping on the windowpane.',
        'target': 'словно крошечные пальцы постукивали по оконному стеклу.',
        'uncommon_words': [{
            'source': 'windowpane',
            'target': 'оконное стекло',
        }],
    }]
})
output

TranslationOutput(sentences=[SentencePair(source='Many years later, Meggie had only to close her eyes', target='Много лет спустя Мегги стоило только закрыть глаза', uncommon_words=[]), SentencePair(source='and she could still hear it,', target='и она всё ещё могла слышать это,', uncommon_words=[]), SentencePair(source='like tiny fingers tapping on the windowpane.', target='словно крошечные пальцы постукивали по оконному стеклу.', uncommon_words=[UncommonWord(source='windowpane', target='оконное стекло')])])

In [5]:
from pydub import AudioSegment
from qwen import generate_audio

instruct = (
    "Speak calmly in a neutral tone. At the end of every sentence "
    "lower your pitch noticeably downwards to signal a complete "
    "pause/closure. Avoid any rising pitch or upward voice inflections "
    "at sentence boundaries."
)
source_lang = "English"
target_lang = "Russian"
export_filename = "wowsuper.mp3"

total = AudioSegment.empty()
for sentence in output.sentences:
    print(f"Generating audio: {(sentence.source, instruct, source_lang)}")
    total += generate_audio(sentence.source, instruct, source_lang)
    print(f"Generating audio: {(sentence.target, instruct, target_lang)}")
    total += generate_audio(sentence.target, instruct, target_lang)
    for word in sentence.uncommon_words or []:
        print(f"Generating audio: {(word.source, instruct, source_lang)}")
        total += generate_audio(word.source, instruct, source_lang)
        print(f"Generating audio: {(word.target, instruct, target_lang)}")
        total += generate_audio(word.target, instruct, target_lang)

total.export(export_filename, format="mp3")
print(f"Written to {export_filename}")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Generating audio: ('Many years later, Meggie had only to close her eyes', 'Speak calmly in a neutral tone. At the end of every sentence lower your pitch noticeably downwards to signal a complete pause/closure. Avoid any rising pitch or upward voice inflections at sentence boundaries.', 'English')


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Generating audio: ('Много лет спустя Мегги стоило только закрыть глаза', 'Speak calmly in a neutral tone. At the end of every sentence lower your pitch noticeably downwards to signal a complete pause/closure. Avoid any rising pitch or upward voice inflections at sentence boundaries.', 'Russian')


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Generating audio: ('and she could still hear it,', 'Speak calmly in a neutral tone. At the end of every sentence lower your pitch noticeably downwards to signal a complete pause/closure. Avoid any rising pitch or upward voice inflections at sentence boundaries.', 'English')


... (6 outputs left out)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Generating audio: ('windowpane', 'Speak calmly in a neutral tone. At the end of every sentence lower your pitch noticeably downwards to signal a complete pause/closure. Avoid any rising pitch or upward voice inflections at sentence boundaries.', 'English')


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Generating audio: ('оконное стекло', 'Speak calmly in a neutral tone. At the end of every sentence lower your pitch noticeably downwards to signal a complete pause/closure. Avoid any rising pitch or upward voice inflections at sentence boundaries.', 'Russian')
Written to wowsuper.mp3


In [8]:
generate_audio("оконное стекло", instruct, "Russian")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [9]:
generate_audio("око\u0301нное стекло", instruct, "Russian")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [14]:
ord("око\u0301нное стекло"[3])

769